# P6 — Synthetic data, fairness, and transportability

The model works *on average, here*. This lesson asks three harder questions:

| Part | Question |
|---|---|
| **A. Synthetic data** | Our dataset is synthetic. How is such data made, is it still useful, and is it really private? |
| **B. Fairness** | Does the model work equally well for girls and boys, infants and older children, HIV+, malnourished, poorer families? |
| **C. Transportability** | Would a model built in Uganda work in another hospital, e.g. your PICU? |
| **D. Reporting** | What does TRIPOD+AI require, and where did we cover it? |

We keep the same 11-predictor logistic model from P3–P5.

⚠️ Synthetic data, so the methods transfer and the exact numbers may not. Subgroups get small fast, so treat every subgroup number with caution.

## 1. Setup
Same predictors and model as P3–P5.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

X = pd.read_csv("p2_features.csv")
y = X.pop("inhospital_mortality").to_numpy()
yes = lambda col, val: (X[col] == val).astype(float).where(X[col].notna())
X["caprefill"] = yes("caprefill_adm", "Yes")
X["respdistress"] = yes("respdistress_adm", "Yes")
X["hiv_pos"] = yes("hivstatus_adm", "HIV positive")
X["malaria_pos"] = yes("malariastatuspos_adm", "Yes")
X["not_feeding"] = (X["feedingstatus_adm"] == "Not feeding at all").astype(float)
NUM = ["bcs_total", "waz", "lactate_mmolpl_adm", "spo2_min", "rr_ratio"]
BIN = ["lactate_measured", "caprefill", "respdistress", "hiv_pos", "malaria_pos", "not_feeding"]
PRED = NUM + BIN

def logreg():
    prep = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUM),
        ("bin", SimpleImputer(strategy="most_frequent"), BIN)])
    return Pipeline([("prep", prep), ("lr", LogisticRegression(C=1e6, max_iter=5000))])

rng = np.random.default_rng(0)
print(len(PRED), "predictors,", y.sum(), "deaths")

# Part A — Synthetic data and privacy

## 2. How our dataset was made
The challenge organisers couldn't share the real Ugandan records, so they released a **synthetic** copy made with `synthpop`'s **sequential CART** method:

1. Draw column 1 (e.g. age) from its real distribution.
2. For column 2, fit a decision tree predicting it from column 1 in the **real** data. For each synthetic row, find its leaf and copy the column-2 value of a **random real child in that leaf**.
3. Repeat for column 3 (using columns 1–2), and so on up to the outcome.

The result keeps the distributions and the main relationships, but each row is a *mash-up* of different real children.

**Our experiment:** treat the challenge data as if it were "real". Split it 70/30, synthesise a copy of the 70% with the same method, then ask:
- **Utility:** does a model trained on the synthetic data work on real hold-out children?
- **Privacy:** are synthetic rows suspiciously close to real children?

`min_samples_leaf` is the key privacy knob: every leaf must pool at least that many real children.

In [ ]:
D = X[PRED].assign(died=y)
real_tr, real_te = train_test_split(D, test_size=0.3, stratify=D["died"], random_state=0)

def cart_synthesize(real, n, min_leaf=10, seed=0):
    r = np.random.default_rng(seed)
    cols = list(real.columns)
    syn = pd.DataFrame(index=range(n), columns=cols, dtype=float)
    syn[cols[0]] = r.choice(real[cols[0]].to_numpy(), n)                      # step 1: marginal
    for j, col in enumerate(cols[1:], 1):
        prev = cols[:j]
        tree = DecisionTreeRegressor(min_samples_leaf=min_leaf, random_state=seed)
        tree.fit(real[prev].fillna(-999), real[col].fillna(-999))            # NaN kept as its own value
        leaf_real = tree.apply(real[prev].fillna(-999))
        leaf_syn = tree.apply(syn[prev].fillna(-999))
        donors = pd.Series(real[col].to_numpy()).groupby(leaf_real).apply(lambda s: s.to_numpy())
        syn[col] = [r.choice(donors[l]) for l in leaf_syn]                   # copy a random donor's value
    return syn

syn = cart_synthesize(real_tr, len(real_tr))
print(f"real training rows: {len(real_tr)}   synthetic rows: {len(syn)}")
pd.DataFrame({"real mean": real_tr.mean(), "synthetic mean": syn.mean()}).round(3)

**What to look for:** the means match almost exactly, mortality included (4.4%). Marginal distributions are the *easy* part. The real test is whether relationships survive.

## 3. Utility: train on synthetic, test on real (TSTR)
The standard utility test is to compare
- **TRTR**: train on real, test on real hold-out, and
- **TSTR**: train on *synthetic*, test on the *same* real hold-out.

**What to look for:**
- TSTR ≈ TRTR (≈ 0.81 vs 0.81). For *this* model the synthetic data is almost as useful as the real data.
- Both are higher than our cross-validated 0.75 from P3. That's **one lucky 30% split** with only about 36 deaths: a reminder of why P3 used 50 folds.
- Caveat: good TSTR for a simple model doesn't guarantee that **rare combinations** (e.g. HIV+ *and* comatose) or interactions were preserved. Synthesis smooths the tails, which is exactly where the sickest children are.

In [ ]:
def auroc_on_real_test(train):
    m = logreg().fit(train[PRED], train["died"].astype(int))
    return roc_auc_score(real_te["died"], m.predict_proba(real_te[PRED])[:, 1])

print(f"TRTR  train real      -> test real: AUROC {auroc_on_real_test(real_tr):.3f}")
print(f"TSTR  train synthetic -> test real: AUROC {auroc_on_real_test(syn):.3f}")

## 4. Privacy: distance to the closest real record (DCR)
"No row is a real child" isn't enough. If a synthetic row is **almost identical** to a real child, someone who knows that child's age, weight and HIV status could find them, and learn they died.

**DCR test:** for each synthetic row, compute the distance (standardised) to the **nearest** real training child. Compare with a fair benchmark: real **hold-out** children, who are *new* people from the same population.
- Synthetic rows much closer than hold-out children → the generator is **memorising**.

We also build a "bad" generator with `min_samples_leaf=1`: each leaf holds a single real child, so it can only copy that child.

**What to look for:**
**Reading the plot:** each curve shows the share of rows (y) lying within distance x of a real training child. A curve further **left** means rows closer to real children, i.e. more privacy risk.

- **min_leaf = 10:** synthetic rows are a little closer than new real children (median ~0.36 vs ~0.42), with about 0.1% exact copies. That's a mild, acceptable signal.
- **min_leaf = 1:** the curve jumps straight to 1 at distance 0: **100% of synthetic rows are exact copies** of real children. It's "synthetic" in name only, and its TSTR utility is still ~0.80, so **good utility tells you nothing about privacy.** Both must be checked.

In [ ]:
def dcr(a, b, ref):
    A = ((a - ref.mean()) / ref.std()).fillna(-9).to_numpy()
    B = ((b - ref.mean()) / ref.std()).fillna(-9).to_numpy()
    return np.array([np.sqrt(((B - r) ** 2).sum(1)).min() for r in A])

syn_leaky = cart_synthesize(real_tr, len(real_tr), min_leaf=1)
d = {"real hold-out (new children)": dcr(real_te[PRED], real_tr[PRED], real_tr[PRED]),
     "synthetic, min_leaf=10": dcr(syn[PRED], real_tr[PRED], real_tr[PRED]),
     "synthetic, min_leaf=1": dcr(syn_leaky[PRED], real_tr[PRED], real_tr[PRED])}

plt.figure(figsize=(8, 4))
for k, v in d.items():                      # cumulative curve: share of rows within distance x
    plt.step(np.sort(v), np.arange(1, len(v) + 1) / len(v), where="post", label=k)
plt.xlim(-0.05, 2); plt.xlabel("Distance to closest real training child (standardised)")
plt.ylabel("Cumulative share of rows"); plt.legend()
plt.tight_layout(); plt.show()

print(f"TSTR AUROC of the min_leaf=1 'synthetic' data: {auroc_on_real_test(syn_leaky):.3f}")
pd.DataFrame({k: {"median DCR": np.median(v), "exact copies %": 100 * np.mean(v == 0)} for k, v in d.items()}).T.round(2)

**Take-home (Part A):** synthetic data is a practical way to share clinical data for **teaching, method development and challenges**, but:
- Check **utility** (TSTR) *and* **privacy** (DCR, exact copies, attribute-disclosure tests). They trade off.
- Formal guarantees need **differential privacy**. CART synthesis has none.
- **Never publish clinical conclusions from synthetic data alone.** Validate on real data (for this challenge, the organisers evaluated submissions on the real hidden test set).

# Part B — Fairness across subgroups

## 5. One model, one threshold, many groups
We use out-of-fold predictions (P4) and a **single global threshold** (the last ROC point with FPR ≤ 20%, which gives an overall FPR of 0.17). Then, for each subgroup, we compute:
- **AUROC with 95% bootstrap CI**: does the model *rank* well within the group?
- **O/E**: observed ÷ expected deaths. > 1 means the model **underestimates** the group's risk.
- **Sensitivity**: the share of the group's deaths that are flagged.
- **FPR**: the share of the group's survivors that are falsely flagged.

*(Takes about 10 seconds.)*

In [ ]:
oof = np.zeros(len(y))
for tr, te in StratifiedKFold(5, shuffle=True, random_state=1).split(X, y):
    oof[te] = logreg().fit(X.iloc[tr][PRED], y[tr]).predict_proba(X.iloc[te][PRED])[:, 1]
fpr, tpr, thr = roc_curve(y, oof)
t20 = thr[np.where(fpr <= 0.20)[0][-1]]
flag = oof >= t20
print(f"Global threshold {t20:.3f}: overall sensitivity {flag[y == 1].mean():.2f}, FPR {flag[y == 0].mean():.2f}")

edu = X["momedu_adm"].replace({"Post secondary (including post S4 technical school)": "post-secondary",
                               "S1-S6": "secondary", "P4-P7": "upper primary", "< = P3": "lower primary",
                               "No school": "no school", "Dont know": "don't know"})
groups = {"sex": X["sex_adm"],
          "age": pd.cut(X["agecalc_adm"], [0, 12, 24, 61], right=False, labels=["6-11 m", "12-23 m", "24-60 m"]),
          "HIV": X["hivstatus_adm"],
          "nutrition": X["sam"].map({0: "no SAM", 1: "SAM"}),
          "malaria": X["malariastatuspos_adm"].map({"Yes": "malaria +", "No": "malaria -"}),
          "mother's education": edu}
rows = []
for g, s in groups.items():
    for lvl in s.dropna().unique():
        m = (s == lvl).to_numpy()
        yy, pp, ff = y[m], oof[m], flag[m]
        boots = [roc_auc_score(yy[b], pp[b]) for b in (rng.integers(0, m.sum(), m.sum()) for _ in range(300))
                 if 0 < yy[b].sum() < len(b)]
        rows.append([g, lvl, m.sum(), yy.sum(), 100 * yy.mean(), roc_auc_score(yy, pp),
                     np.percentile(boots, 2.5), np.percentile(boots, 97.5),
                     yy.mean() / pp.mean(), ff[yy == 1].mean(), ff[yy == 0].mean()])
sub = pd.DataFrame(rows, columns=["group", "level", "n", "deaths", "mort %", "AUROC", "lo", "hi", "O/E", "sens", "FPR"])
sub.round(2)

## 6. Seeing it: AUROC with CIs and the error trade-off by group
**What to look for:**
- **Left (AUROC ± 95% CI):** most intervals overlap the overall ~0.75, so we can't claim a real difference. The small groups (HIV+ 80 children / 8 deaths, "no school" 100 / 4 deaths) have intervals so wide they're **uninformative**. Absence of evidence isn't evidence of fairness.
- **Right (sensitivity vs FPR at the same threshold):** this is where real differences appear.
  - **HIV+:** sensitivity 0.88 but **FPR 0.67**. Two-thirds of HIV+ survivors are flagged, because HIV itself carries OR 2.2 and pushes every HIV+ child over the line.
  - **SAM:** FPR 0.40 vs 0.13 without SAM. Malnourished children bear most of the false alarms.
  - **Infants 6–11 m:** sensitivity only **0.45** vs 0.69 at 24–60 m. The model misses more infant deaths, and O/E ≈ 0.8 means it *over*-predicts their risk. Infant physiology differs, and our age-adjusted features may not fully capture it.
  - **Mother's education, a social gradient:** sensitivity falls steadily from **0.91** (post-secondary) → 0.57 (secondary) → 0.56 → 0.44 → **0.25** (no school). Each group is small, but the trend is monotone. Deaths among children of the least-educated mothers are the most likely to be missed, probably because these children arrive later or with presentations our predictors capture less well. An unmonitored model could **widen an existing inequity**.
  - **Sex:** O/E 1.15 for girls vs 0.88 for boys. The model somewhat under-predicts girls' risk. The CIs are wide, but this is worth monitoring.

In [ ]:
sub["label"] = sub["group"] + ": " + sub["level"].astype(str)
s2 = sub.iloc[::-1]
fig, ax = plt.subplots(1, 2, figsize=(15, 7), sharey=True)
ax[0].errorbar(s2["AUROC"], s2["label"], xerr=[s2["AUROC"] - s2["lo"], s2["hi"] - s2["AUROC"]], fmt="o", capsize=3)
ax[0].axvline(roc_auc_score(y, oof), ls="--", c="grey"); ax[0].axvline(.5, c="k", lw=.6)
ax[0].set(xlabel="AUROC (95% bootstrap CI)", title="Discrimination within each subgroup")
ax[1].scatter(s2["sens"], s2["label"], label="sensitivity", marker="o")
ax[1].scatter(s2["FPR"], s2["label"], label="false-positive rate", marker="x", c="red")
ax[1].axvline(.2, ls=":", c="red"); ax[1].set(xlabel="Rate at the single global threshold", title="Who gets flagged?")
ax[1].legend(loc="lower right")
plt.tight_layout(); plt.show()

## 7. What *is* fair? The impossibility problem
There are several reasonable definitions of fairness:

| Criterion | Meaning | Here |
|---|---|---|
| **Calibration within groups** | A 10% risk means 10% in every group | Roughly (O/E 0.8–1.2) |
| **Equal opportunity** | Same **sensitivity** in each group | ✗ infants 0.45 vs older 0.69 |
| **Equal false-alarm burden** | Same **FPR** in each group | ✗ HIV+ 0.67 vs HIV− 0.16 |

When **base rates differ** (HIV+ mortality 10% vs 4.3%), a model **can't satisfy all three at once**. This is mathematically proven (Kleinberg et al. 2016; Chouldechova 2017). Choosing which one matters is a **clinical and ethical decision**, not a technical one:
- For a *cheap* action (extra observation), a high FPR in HIV+ children may be fine: they *are* at higher risk.
- For a *scarce* resource (one ICU bed), it concentrates the resource by diagnosis. Is that what you want?

**What to do in practice:** report subgroup performance (TRIPOD+AI item), involve clinicians and families in choosing the criterion, consider group-specific thresholds where justified, and **collect more data** for small groups rather than drawing conclusions from 4 deaths.

# Part C — Transportability: would this work in *your* PICU?

## 8. Prevalence shift → recalibrate the intercept
Your unit's mortality differs from 4.4%. We simulate two hospitals by resampling survivors and deaths:
- **Hospital L**: mortality 1.5% (e.g. a well-resourced PICU with less severe sepsis)
- **Hospital H**: mortality 10% (e.g. a referral centre with sicker children)

**Fix: intercept recalibration.** Keep all coefficients and shift the baseline log-odds by one number *c*, estimated from a local sample (here half of each hospital's children), then check it on the other half. It's the cheapest local adaptation and needs only outcome counts.

**What to look for:**
- Hospital L: the model predicts ~4% while ~1.3% die (**3× overestimate**). Hospital H: predicts ~5% while 10% die (**2× underestimate**).
- The **AUROC differs** between the two (~0.85 vs ~0.74), but that's resampling noise. At 1.5% mortality only ~15 deaths remain in the evaluation half. Ranking itself doesn't depend on prevalence.
- After adding *c* (≈ −1.1 or +0.85), mean predicted ≈ observed.

⚠️ Uncorrected, Hospital L's clinicians would get 3× too many alerts, and alert fatigue follows.

In [ ]:
logit = lambda p: np.log(p / (1 - p))
sig = lambda z: 1 / (1 + np.exp(-z))
res = []
for name, target in [("Hospital L", 0.015), ("Hospital H", 0.10)]:
    n_d = int(round(target * 2000))
    idx = np.concatenate([rng.choice(np.where(y == 1)[0], n_d), rng.choice(np.where(y == 0)[0], 2000 - n_d)])
    yy, pp = y[idx], oof[idx]
    half = rng.permutation(len(idx)); a, b = half[:1000], half[1000:]
    c = brentq(lambda c: (yy[a] - sig(logit(pp[a]) + c)).sum(), -5, 5)   # local intercept update
    res.append({"site": name, "observed mortality": yy[b].mean(), "mean predicted (original)": pp[b].mean(),
                "AUROC": roc_auc_score(yy[b], pp[b]), "intercept shift c": c,
                "mean predicted (recalibrated)": sig(logit(pp[b]) + c).mean()})
pd.DataFrame(res).set_index("site").round(3)

## 9. Practice shift: when a feature means something different
`lactate_measured` (OR 1.51) encodes **Ugandan testing practice**: sicker-looking children got lactate. Imagine deploying in a PICU that measures lactate in **every** child.

**What to look for:** the children who were *not* tested in Uganda (observed mortality 3.0%) jump from a predicted **3.0% to 4.3%** (+43%) just because the flag switched on. Their illness hasn't changed. The model is now miscalibrated for them, and the "test ordered" signal it relies on has vanished.

This is the general risk with **process variables**: order patterns, oxygen use (P1), SpO₂ taken on oxygen (P3), device signal quality (P3). They carry information about *clinician behaviour at the development site*. At a new site, either drop them or re-fit locally.

In [ ]:
model = logreg().fit(X[PRED], y)
p_all = model.predict_proba(X[PRED])[:, 1]
X_newsite = X[PRED].copy(); X_newsite["lactate_measured"] = 1.0
unm = X["lactate_measured"].to_numpy() == 0
print(f"Children NOT tested in Uganda (n={unm.sum()}): observed mortality {y[unm].mean():.3f}")
print(f"  predicted, Ugandan practice   : {p_all[unm].mean():.3f}")
print(f"  predicted, 'test everyone' site: {model.predict_proba(X_newsite)[:, 1][unm].mean():.3f}")

## 10. Case-mix shift: train on one population, test on another
A quick "internal–external" check: develop on children **< 24 months** and test on children **≥ 24 months**, a population the model has never seen.

**What to look for:** AUROC ≈ 0.77 and mean predicted ≈ observed (~4.5%), so this model **transports reasonably across age**. That's good news, and it only counts because we *tested* it. The real test for *your* PICU is different geography, case mix (surgical and cardiac children, not only sepsis), resources, and a different mortality. Uganda → Saudi Arabia is a much bigger jump than < 2 y → ≥ 2 y.

**Before using any external model locally:**
1. **External validation** on a local retrospective cohort: discrimination, calibration, decision curve.
2. **Recalibrate** (intercept, or intercept + slope). Re-fit if calibration is badly off.
3. **Silent deployment:** run the model in the background for months and compare alerts with outcomes.
4. **Monitor drift** after go-live. Practice changes such as a new sepsis protocol change what the features mean.

In [ ]:
young = (X["agecalc_adm"] < 24).to_numpy()
m_young = logreg().fit(X[young][PRED], y[young])
p_old = m_young.predict_proba(X[~young][PRED])[:, 1]
print(f"Develop on <24 m : n={young.sum()}, mortality {y[young].mean():.3f}")
print(f"Test on  >=24 m  : n={(~young).sum()}, mortality {y[~young].mean():.3f}, "
      f"AUROC {roc_auc_score(y[~young], p_old):.3f}, mean predicted {p_old.mean():.3f}")

# Part D — Reporting: TRIPOD+AI (BMJ 2024)
TRIPOD+AI is the reporting guideline for prediction models using regression or machine learning. Here is how the peds track maps onto its core items:

| TRIPOD+AI area | What it asks | Where we did it |
|---|---|---|
| Data source & participants | Setting, eligibility, dates, flow | P1 (6 Ugandan hospitals, 6–60 m, n = 2,686) |
| Outcome | Definition, timing, blinding | P1 (in-hospital death; `lengthadm` is an outcome, not a predictor) |
| Predictors | Definition, timing, **availability at prediction time** | P2 (leakage removal, admission-only) |
| Missing data | How much, how handled | P1 §6, P2 (flags), P3 (imputation inside the pipeline) |
| Sample size | Justification (EPV, Riley et al.) | P1, P3 (EPV ≈ 11) |
| Model building | Selection, tuning, handling non-linearity | P3 (clinical choice), P5 (boosting, splines) |
| Class imbalance | Any rebalancing, and its effect on calibration | P4 (don't rebalance) |
| Performance | Discrimination **with CIs**, **calibration**, clinical utility | P3, P4 (calibration, DCA), P5 (bootstrap CIs) |
| Fairness | Performance in key subgroups | **P6 Part B** |
| Interpretability | How predictions can be explained | P3 (ORs), P5 (SHAP) |
| Internal / external validation | Method, **transportability** | P3–P5 (repeated CV), **P6 Part C** |
| Data sharing & privacy | Synthetic data, access | **P6 Part A** |
| Limitations | Synthetic data, small subgroups, single country | here ↓ |

**Limitations of everything we built:** synthetic training data; only 119 deaths (wide CIs); one country and one disease pathway; process variables that won't transport; no prospective or real-world evaluation.

## Summary
| Part | Key result |
|---|---|
| Synthetic data | TSTR 0.81 ≈ TRTR 0.81; min_leaf=10 gives 0.1% exact copies, min_leaf=1 gives **100% copies with good utility** |
| Fairness | Overlapping AUROC CIs, but at one threshold: HIV+ FPR 0.67, SAM FPR 0.40, infant sensitivity 0.45; sensitivity falls with maternal education (0.91 → 0.25) |
| Prevalence shift | 3× over- or 2× under-prediction; fixed by a one-number intercept update |
| Practice shift | "Test everyone" inflates risk +43% for previously untested children |
| Case-mix shift | < 2 y → ≥ 2 y transports well (AUROC 0.77, calibrated), but only because we checked |

## ✍️ Try it yourself
1. **Privacy knob:** run `cart_synthesize` with `min_leaf=3` and `min_leaf=50`. Plot TSTR AUROC against the % of exact copies. Where would you set it?
2. **Group thresholds:** choose a separate threshold for HIV+ children that brings their FPR down to 0.20. What happens to their sensitivity? Is that acceptable?
3. **Slope recalibration:** in §8, fit `LogisticRegression().fit(logit(pp[a]).reshape(-1, 1), yy[a])` (intercept *and* slope). Does it beat intercept-only? (With few deaths it often doesn't. That's why intercept-only is the usual first step.)
4. **Drop the process variable:** refit the model without `lactate_measured`. How much AUROC do you lose? Would you trade that for better transportability?
5. **Your PICU:** list the 11 predictors. Which are routinely recorded at admission in your unit, which are measured differently (e.g. Blantyre vs GCS/AVPU), and which don't apply?

🎓 **This completes the paediatric track.** Next options: apply for credentialed access to the PIC paediatric ICU database, or plan a local validation study for your PICU.